In [12]:
import argparse
import logging
import os, sys
import torch
import numpy as np
import pandas as pd 
import random
import json

from pathlib import Path

# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo


def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

def init_cfg(folder):
    cfg = Config.fromfile(f'{folder}/vfnet_r18.py')
    return cfg


def get_best_configs():
    df1 = pd.read_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/table_Single.pkl')
    df2 = pd.read_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/table_Multi.pkl')
    
    return {'single':df1, 'multi':df2}


setup_cache_size_limit_of_dynamo()

single, multi = get_best_configs()['single'], get_best_configs()['multi']


In [13]:
def findWeight(folder):
    pth = os.listdir(folder)
    pth = [x for x in pth if x.endswith('.pth')]
    pth = [x for x in pth if x.startswith('epoch')]
    assert len(pth) == 1, f'Found {len(pth)} weights in {folder}'
    return f'{folder}/{pth[0]}'

In [14]:
def set_NoiseTesting_config(band_sel, single, multi, severity=0, corruption='gaussian', Seed=18):
    """
    Set the configuration for noise testing.
    Args:
        band_sel (int): The selected band index.
        single (DataFrame): The single band information.
        multi (DataFrame): The multi band information.
        severity (int, optional): The severity of the corruption. Defaults to 0.
        corruption (str, optional): The type of corruption. Defaults to 'gaussian'.
    Returns:
        Config: The modified configuration object.
    """
    

    CORRUPTION = corruption
    SEVERITY = severity
    IMG_SIZE = 2048 # Default
    
    
    # Init Config
    _, LR, BS, ME = single.index[band_sel-1]
    base = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS'
    cfgDir = f'{base}/Single/perfect_b{band_sel}/{Seed}_BS_{BS}_LR_{LR}_ME_{ME}_OPT_SGD'
    cfg = init_cfg(folder=f'{cfgDir}')
    

    # Modify Config
    CHECKPOINT = findWeight(f'{cfgDir}')
    cfg.load_from = CHECKPOINT
    cfg.test_dataloader.dataset.pipeline = [{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': [1]},
                        dict(type='LoadAnnotations', with_bbox=True),
                        dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                        dict(type='ImageCorruption', corruption=CORRUPTION, severity=SEVERITY), # 'gaussian', 'salt', 'pepper', 's&p' (salt and pepper), 'speckle', 'poisson'
                        dict(
                            meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                    'height', 'width', 'instances', 'sample_idx', 
                                    'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                    'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                    'gt_bboxes_labels'),
                            type='PackDetInputs'),
                    ]

    work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
    cfg.work_dir = work_dir
    
    return cfg


In [ ]:
# cfg = set_NoiseTesting_config(band_sel=1, single=single, multi=multi, severity=4, corruption='gaussian')

# # Run the test
# runner = RUNNERS.build(cfg)
# work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
# runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
# output_test_data = runner.test()

# Cycle Loop

In [15]:
single

,,,,coco/bbox_mAP,coco/bbox_mAP_50,coco/bbox_mAP_75,coco/bbox_mAP_s,coco/bbox_mAP_m,coco/bbox_mAP_l
Band,LR,BS,ME,,,,,,
1,0.0009,2,30,0.1656,0.4234,0.0954,0.1656,-1.0,-1.0
2,0.0009,2,30,0.2852,0.6576,0.2082,0.2852,-1.0,-1.0
3,0.001,2,30,0.3295,0.6925,0.2780,0.3295,-1.0,-1.0
4,0.0009,2,30,0.3826,0.7252,0.3588,0.3828,-1.0,-1.0
5,0.001,2,30,0.5155,0.8155,0.5870,0.5155,-1.0,-1.0
6,0.0008,2,30,0.4536,0.7822,0.4928,0.4536,-1.0,-1.0
7,0.0009,2,30,0.4584,0.7848,0.4940,0.4584,-1.0,-1.0
8,0.0008,2,30,0.4646,0.7960,0.5062,0.4646,-1.0,-1.0
9,0.0008,2,30,0.4742,0.7920,0.5254,0.4742,-1.0,-1.0


In [16]:
Results = {'Band':[], 'Severity':[], 'AP':[], 'AP50':[], 'AP75':[]}

for band_sel in range(1, 13, 1):
    for S in np.arange(0, 5.5, 0.5):
        try:
            cfg = set_NoiseTesting_config(band_sel=band_sel, single=single, multi=multi, severity=S, corruption='gaussian', Seed=53)

            # Run the test
            runner = RUNNERS.build(cfg)
            work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
            runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
            output_test_data = runner.test()
        
        except Exception as e:
            print(f'Error in band {band_sel} and severity {S}')
            print(f'Trying with another seed..')
            
            cfg = set_NoiseTesting_config(band_sel=band_sel, single=single, multi=multi, severity=S, corruption='gaussian', Seed=71)
            # Run the test
            runner = RUNNERS.build(cfg)
            work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
            runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
            output_test_data = runner.test()

        
        
        Results['Band'].append(band_sel)
        Results['Severity'].append(S)
        Results['AP'].append(output_test_data['coco/bbox_mAP'])
        Results['AP50'].append(output_test_data['coco/bbox_mAP_50'])
        Results['AP75'].append(output_test_data['coco/bbox_mAP_75'])

08/15 08:59:25 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 53
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/15 08:59:39 - mmengine - INFO - Epoch(test) [ 3/57]    eta: 0:01:52  time: 2.0810  data_time: 0.2354  memory: 1112  
08/15 08:59:39 - mmengine - INFO - Epoch(test) [ 4/57]    eta: 0:01:26  time: 1.6334  data_time: 0.2023  memory: 1112  
08/15 08:59:39 - mmengine - INFO - Epoch(test) [ 5/57]    eta: 0:01:10  time: 1.3505  data_time: 0.1687  memory: 1112  
08/15 08:59:39 - mmengine - INFO - Epoch(test) [ 6/57]    eta: 0:00:58  time: 1.1524  data_time: 0.1459  memory: 1112  
08/15 08:59:40 - mmengine - INFO - Epoch(test) [ 7/57]    eta: 0:00:51  time: 1.0222  data_time: 0.1302  memory: 1112  
08/15 08:59:40 - mmengine - INFO - Epoch(test) [ 8/57]    eta: 0:00:45  time: 0.9264  data_time: 0.1202  memory: 1112  
08/15 08:59:40 - mmengine - INFO - Epoch(test) [ 9/57]    eta: 0:00:40  time: 0.8491  data_time: 0.1104  memory: 1112  
08/15 08:59:40 - mmengine - INFO - Epoch(test) [10/57]    eta: 0:00:37  time: 0.7897  data_time: 0.1052  memory: 1112  
08/15 08:59:41 - mmengine - INFO - Epoch

In [ ]:
pd.DataFrame(Results).to_pickle('/Data_large/marine/PythonProjects/MMDET/notebooks/plotters/VENuS/NoiseTesting.pkl')

# Plot

In [ ]:
from style import set_style
from matplotlib import pyplot as plt

set_style()


df = pd.DataFrame(Results)


def get_metrics_x_band(df, band):
    df = df[df['Band'] == band]
    return df




b1 = get_metrics_x_band(df, 1)


fig = plt.figure()
for band in range(1, 13, 1):
    
    plt.subplot(2,6,band)

    b = get_metrics_x_band(df, band)
    x = b['Severity'].to_numpy()
    y = b['AP50'].to_numpy()
    
    plt.plot(b['Severity'], b['AP50'], label=f'Band {band}')
    plt.legend()

plt.show()
